# TauDEM to HMS Atlas 14 Bootstrap

This notebook demonstrates a full Spring Creek workflow using the committed TauDEM fixture, a cloned `river_bend` HMS shell, parser-of-record validation through HEC-HMS, and a compute-ready Atlas 14 frequency storm run.

In [1]:
# pip install hms-commander

**For Development**: If working on hms-commander source code, use the `hmscmdr_local` conda environment (editable install) instead of pip install.

## Setup

Resolve the repository root, the committed Spring Creek TauDEM fixture, the deterministic NOAA PFDS fixture, and a disposable output directory for the Atlas 14 bootstrap.

In [2]:
from pathlib import Path
import json
import shutil

import pandas as pd

from hms_commander import (
    Atlas14Storm,
    HmsBasinBuilder,
    HmsCmdr,
    HmsExamples,
    HmsJython,
    HmsPrj,
    HmsRoundTripValidator,
)

cwd = Path.cwd()
repo_root = cwd if (cwd / "hms_commander").exists() else cwd.parent
assert (repo_root / "hms_commander").exists(), f"Could not resolve repo root from {cwd}"

fixture_root = repo_root / "tests" / "fixtures" / "taudem_spring_creek"
atlas14_fixture = repo_root / "tests" / "fixtures" / "atlas14" / "spring_creek_pfds_depth_english_pds.txt"
example_projects_root = repo_root / "examples" / "hms_example_projects"
workspace_root = repo_root / "examples" / "output" / "21_taudem_to_hms_atlas14"
output_dir = workspace_root / "spring_creek_atlas14_bootstrap"
workspace_root.mkdir(parents=True, exist_ok=True)
if output_dir.exists():
    shutil.rmtree(output_dir)

pd.Series({
    "repo_root": str(repo_root),
    "fixture_root": str(fixture_root),
    "atlas14_fixture": str(atlas14_fixture),
    "workspace_root": str(workspace_root),
})

repo_root                                        G:\GH\hms-commander
fixture_root       G:\GH\hms-commander\tests\fixtures\taudem_spri...
atlas14_fixture    G:\GH\hms-commander\tests\fixtures\atlas14\spr...
workspace_root     G:\GH\hms-commander\examples\output\21_taudem_...
dtype: str

## Extract the HMS Shell

Use `HmsExamples.extract_project()` to stage a reproducible `river_bend` shell. The TauDEM import is written into a fresh clone of that project shape.

In [3]:
river_bend_dir = HmsExamples.extract_project(
    "river_bend",
    version="4.13",
    output_path=example_projects_root,
)

pd.Series({
    "river_bend_dir": str(river_bend_dir),
    "exists": river_bend_dir.exists(),
})

2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Found HMS 4.10 at C:\Program Files\HEC\HEC-HMS\4.10


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Found HMS 4.11 at C:\Program Files\HEC\HEC-HMS\4.11


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Found HMS 4.12 at C:\Program Files\HEC\HEC-HMS\4.12


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Found HMS 4.13 at C:\Program Files\HEC\HEC-HMS\4.13


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Found HMS 4.4.1 at C:\Program Files\HEC\HEC-HMS\4.4.1


...

2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Extracting 'river_bend' from HMS 4.13


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Source: C:\Program Files\HEC\HEC-HMS\4.13\samples.zip


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Destination: G:\GH\hms-commander\examples\hms_example_projects\river_bend


2026-04-23 18:34:51 - hms_commander.HmsExamples - INFO - Successfully extracted 'river_bend' to G:\GH\hms-commander\examples\hms_example_projects\river_bend


river_bend_dir    G:\GH\hms-commander\examples\hms_example_proje...
exists                                                         True
dtype: object

## Build the TauDEM-to-HMS Spec

Assemble the durable TauDEM handoff spec: one reach per `LINKNO`, one subbasin per `WSNO`, junctions from TauDEM connectivity, and the verified boundary handoff outlet as the terminal sink.

In [4]:
spec = HmsBasinBuilder.build_taudem_hms_spec(fixture_root, unit_system="English")

pd.DataFrame([
    {"metric": "subbasins", "value": spec["counts"]["subbasins"]},
    {"metric": "reaches", "value": spec["counts"]["reaches"]},
    {"metric": "junctions", "value": spec["counts"]["junctions"]},
    {"metric": "sink_x", "value": spec["sink"]["x"]},
    {"metric": "sink_y", "value": spec["sink"]["y"]},
    {"metric": "crs", "value": spec["crs"]["epsg"]},
    {"metric": "unit_system", "value": spec["unit_system"]},
])

2026-04-23 18:34:51 - rasterio._env - WARNING - CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


,metric,value
0,subbasins,60
1,reaches,60
2,junctions,29
3,sink_x,531443.987
4,sink_y,1883487.629
5,crs,EPSG:5070
6,unit_system,English


## Parse the Atlas 14 Point-Frequency Fixture

For deterministic documentation, this notebook uses the committed NOAA PFDS response fixture captured for the Spring Creek gauge location. In live workflows, swap this for `Atlas14Storm.get_frequency_storm_depths(...)`.

In [5]:
pfds_response = atlas14_fixture.read_text(encoding="utf-8")
parsed_pfds = Atlas14Storm.parse_pfds_response(pfds_response)
depth_table = Atlas14Storm.build_depth_duration_table(parsed_pfds)
frequency_depths_100yr = Atlas14Storm.build_frequency_storm_depths(depth_table, ari_years=100)

depth_summary = depth_table[depth_table["duration_minutes"].isin(Atlas14Storm.FREQUENCY_STORM_DURATIONS_MIN)].copy()
depth_summary = depth_summary[["duration_label", "duration_minutes", 100]]
depth_summary["hms_frequency_depth_inches"] = frequency_depths_100yr
depth_summary

,duration_label,duration_minutes,100,hms_frequency_depth_inches
0,5-min,5,0.861,0.861
2,15-min,15,1.610,1.610
3,30-min,30,2.320,2.320
4,60-min,60,3.110,3.110
5,2-hr,120,3.780,3.780
6,3-hr,180,4.100,4.100
7,6-hr,360,4.810,4.810
9,24-hr,1440,6.150,6.150


## Bootstrap the Atlas 14 HMS Project

Write the TauDEM-derived basin, SQLite geometry, Atlas 14 meteorology, control spec, run file, and project registry entries into a cloned `river_bend` project shell.

In [6]:
paths = HmsBasinBuilder.bootstrap_taudem_atlas14_project(
    spec=spec,
    template_project_dir=river_bend_dir,
    output_dir=output_dir,
    basin_name="Spring Creek TauDEM Sink",
    met_name="Spring Creek Atlas14 100YR",
    control_name="Spring Creek Atlas14 Control",
    run_name="Spring Creek Atlas14 Run",
    atlas14_depth_table=depth_table,
    atlas14_latitude=39.815328,
    atlas14_longitude=-89.698713,
)

pd.Series({name: str(path) for name, path in paths.items()})

2026-04-23 18:34:53 - hms_commander.HmsPrj - INFO -   Gridded model: 3 .sqlite files found


2026-04-23 18:34:53 - hms_commander.HmsPrj - INFO - HMS project initialized: river_bend


2026-04-23 18:34:53 - hms_commander.HmsPrj - INFO -   Version: 4.13


2026-04-23 18:34:53 - hms_commander.HmsPrj - INFO -   Basin models: 3


2026-04-23 18:34:53 - hms_commander.HmsPrj - INFO -   Met models: 1


...

2026-04-23 18:34:54 - pyogrio._io - INFO - Created 60 records


2026-04-23 18:34:54 - pyogrio._io - INFO - Created 60 records


2026-04-23 18:34:54 - pyogrio._io - INFO - Created 29 records


2026-04-23 18:34:54 - pyogrio._io - INFO - Created 1 records


project_dir         G:\GH\hms-commander\examples\output\21_taudem_...
project_file        G:\GH\hms-commander\examples\output\21_taudem_...
basin               G:\GH\hms-commander\examples\output\21_taudem_...
sqlite              G:\GH\hms-commander\examples\output\21_taudem_...
met                 G:\GH\hms-commander\examples\output\21_taudem_...
control             G:\GH\hms-commander\examples\output\21_taudem_...
run                 G:\GH\hms-commander\examples\output\21_taudem_...
bootstrap_report    G:\GH\hms-commander\examples\output\21_taudem_...
atlas14_report      G:\GH\hms-commander\examples\output\21_taudem_...
dtype: str

In [7]:
atlas14_report = json.loads(paths["atlas14_report"].read_text(encoding="utf-8"))

pd.Series({
    "query_source": atlas14_report["query_point"]["source"],
    "atlas14_series": atlas14_report["atlas14"]["series"],
    "atlas14_units": atlas14_report["atlas14"]["units"],
    "return_period_years": atlas14_report["atlas14"]["ari_years"],
    "depths_inches": atlas14_report["atlas14"]["depths_inches"],
})

query_source                                              user_supplied
atlas14_series                                                      pds
atlas14_units                                                   english
return_period_years                                                 100
depths_inches          [0.861, 1.61, 2.32, 3.11, 3.78, 4.1, 4.81, 6.15]
dtype: object

## Round-Trip Validation Through HMS

Use the clone-first parser-of-record validator before computing. This opens the bootstrapped project in HEC-HMS, saves all project components, exits, and checks for unclassified rewrites.

In [8]:
hms_exe = HmsJython.find_hms_executable(version="4.13")
if hms_exe is None:
    roundtrip_result = {"status": "skipped", "reason": "Local HEC-HMS 4.13 not available"}
else:
    roundtrip_result = HmsRoundTripValidator.roundtrip_open_save_close(output_dir, hms_exe_path=hms_exe)

pd.Series(roundtrip_result)

2026-04-23 18:34:54 - hms_commander.HmsJython - INFO - Found HEC-HMS 4.13: C:\Program Files\HEC\HEC-HMS\4.13


2026-04-23 18:34:54 - hms_commander.HmsJython - INFO - Executing HMS 4.13 via direct Java invocation


2026-04-23 18:34:54 - hms_commander.HmsJython - INFO - Script: G:\GH\hms-commander\examples\output\21_taudem_to_hms_atlas14\spring_creek_atlas14_bootstrap\_roundtrip_validation\2026-04-23T22-34-54Z\hms_script.py


2026-04-23 18:34:54 - hms_commander.HmsJython - INFO - Memory: -Xms128M -Xmx4G


2026-04-23 18:34:56 - hms_commander.HmsJython - INFO - HMS 4.13 script executed successfully


...

2026-04-23 18:34:56 - hms_commander.HmsSqlite - INFO - Found 6 spatial layers in Spring_Creek_TauDEM_Sink.sqlite


2026-04-23 18:34:56 - hms_commander.HmsSqlite - INFO - CRS WKT found in Spring_Creek_TauDEM_Sink.sqlite (660 chars)


2026-04-23 18:34:56 - hms_commander.HmsSqlite - INFO - Found 6 spatial layers in Spring_Creek_TauDEM_Sink.sqlite


2026-04-23 18:34:56 - hms_commander.HmsSqlite - INFO - CRS WKT found in Spring_Creek_TauDEM_Sink.sqlite (660 chars)


project_dir               G:\GH\hms-commander\examples\output\21_taudem_...
project_name                                                     river bend
validation_project_dir    G:\GH\hms-commander\examples\output\21_taudem_...
hms_install_path                          C:\Program Files\HEC\HEC-HMS\4.13
artifact_dir              G:\GH\hms-commander\examples\output\21_taudem_...
before_snapshot           G:\GH\hms-commander\examples\output\21_taudem_...
after_snapshot            G:\GH\hms-commander\examples\output\21_taudem_...
headless_success                                                       True
normalized_passed                                                     False
passed                                                                False
execution_error                                                        None
gui_smoke                   {'requested': False, 'status': 'not_requested'}
stdout_path               G:\GH\hms-commander\examples\output\21_taudem_...
stderr_path 

## Compute the Atlas 14 Run

If a local HEC-HMS 4.13 installation is available, initialize the project and run the new Atlas 14 design storm.

In [9]:
if hms_exe is None:
    compute_summary = {"status": "skipped", "reason": "Local HEC-HMS 4.13 not available"}
else:
    project = HmsPrj().initialize(output_dir, hms_exe_path=hms_exe)
    compute_success = HmsCmdr.compute_run("Spring Creek Atlas14 Run", hms_object=project)
    compute_summary = {
        "status": "completed" if compute_success else "failed",
        "success": bool(compute_success),
        "dss_exists": (output_dir / "Spring_Creek_Atlas14_Run.dss").exists(),
        "log_exists": (output_dir / "Spring_Creek_Atlas14_Run.log").exists(),
        "met_exists": paths["met"].exists(),
        "control_exists": paths["control"].exists(),
        "run_exists": paths["run"].exists(),
    }

pd.Series(compute_summary)

2026-04-23 18:34:57 - hms_commander.HmsPrj - INFO -   Gridded model: 4 .sqlite files found


2026-04-23 18:34:57 - hms_commander.HmsPrj - INFO - HMS project initialized: river_bend


2026-04-23 18:34:57 - hms_commander.HmsPrj - INFO -   Version: 4.13


2026-04-23 18:34:57 - hms_commander.HmsPrj - INFO -   Basin models: 4


2026-04-23 18:34:57 - hms_commander.HmsPrj - INFO -   Met models: 2


...

2026-04-23 18:34:57 - hms_commander.HmsJython - INFO - Script: G:\GH\hms-commander\examples\output\21_taudem_to_hms_atlas14\spring_creek_atlas14_bootstrap\hms_script.py


2026-04-23 18:34:57 - hms_commander.HmsJython - INFO - Memory: -Xms128M -Xmx4G


2026-04-23 18:35:02 - hms_commander.HmsJython - INFO - HMS 4.13 script executed successfully


2026-04-23 18:35:02 - hms_commander.HmsCmdr - INFO - Run 'Spring Creek Atlas14 Run' completed successfully


status            completed
success                True
dss_exists             True
log_exists             True
met_exists             True
control_exists         True
run_exists             True
dtype: object

## Next Steps

- Promote this same bootstrap path into downstream `ras-agent` orchestration once the HMS import and compute loop stays stable.
- Extend the writer from topology and geometry into production-ready hydrologic parameter population in a later phase.
- For live Atlas 14 runs, replace the committed PFDS fixture with `Atlas14Storm.get_frequency_storm_depths(...)` using the desired basin query point.